In [1]:
# ==============================================================================
# Encabezado del archivo: LDO_PVT_PEX.ipynb
# Descripción:
#   Este cuaderno de Jupyter automatiza el barrido de esquinas PVT (Proceso,
#   Voltaje y Temperatura) para el regulador LDO en simulación post-layout (PEX)
#   a partir del netlist '/headless/.xschem/simulations/ldo_tb_pex.spice'.
#   Conserva la misma estructura que 'LDO_PVT (Copiar).ipynb', adaptada para
#   omitir el análisis AC de estabilidad, extraer el eje de frecuencia desde
#   'LDO_ac_psrr.raw' y guardar todos los reportes, gráficas y archivos .pkl/.raw
#   con sufijo '_pex' sin alterar ningún archivo original del esquemático.
# ==============================================================================

import site
import sys
import subprocess
import os

# 1. Registrar la ruta de paquetes del usuario en el sistema
user_site = '/headless/.local/lib/python3.12/site-packages'
if os.path.exists(user_site):
    site.addsitedir(user_site)

# 2. Intentar importar PyLTSpice; si falla, instalarlo automáticamente
try:
    from PyLTSpice import RawRead
    print('¡PyLTSpice ya estaba instalado y listo para usar!')
except ImportError:
    print('PyLTSpice no fue encontrado. Instalando automáticamente...')
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '--user', 'PyLTSpice'])
    if os.path.exists(user_site):
        site.addsitedir(user_site)
    from PyLTSpice import RawRead
    print('¡PyLTSpice se instaló e importó correctamente!')


¡PyLTSpice ya estaba instalado y listo para usar!


In [2]:
# ==============================================================================
# Celda 2: Función de simulación individual PVT en paralelo para PEX
# Descripción:
#   Modifica dinámicamente la esquina de proceso (.lib), temperatura (.option temp)
#   y tensión de alimentación (.param VDD) en una copia temporal del netlist PEX,
#   redirige los archivos .raw a una subcarpeta aislada por hilo, ejecuta Ngspice
#   en modo batch y extrae tanto las métricas escalares del log como las curvas
#   de los archivos .raw (omitiendo LDO_ac_estability.raw).
# ==============================================================================

import os
import re
import subprocess
import shutil
import concurrent.futures
import itertools
from PyLTSpice import RawRead
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np

# Diccionarios globales donde se almacenan las métricas y vectores de cada esquina
parametros_ldo = {}
graficas_ldo = {}

def one_pvt_parallel(netlist_path, raw_path, temp, vdd, proces):
    # Nombre de la carpeta limpio y seguro para identificar la esquina PVT
    vdd_str = str(vdd).replace('.', '_')
    nombre_llave = f"V_{vdd_str}_T_{temp}_{proces}"
    
    # 1. CREAR LA CARPETA AISLADA PARA ESTE HILO
    sim_dir = os.path.join(raw_path, nombre_llave)
    os.makedirs(sim_dir, exist_ok=True)
    
    directorio_original = os.path.dirname(netlist_path)
    
    with open(netlist_path, 'r') as archivo:
        contenido = archivo.read()

    # Modificar parámetros PVT (esquina de proceso, temperatura y voltaje VDD)
    patron_corner = r"(\.lib\s+/foss/pdks/sky130A/libs\.tech/ngspice/sky130\.lib\.spice\s+)\w+"
    contenido = re.sub(patron_corner, rf"\g<1>{proces}", contenido)
    
    patron_temp = r"(\.option\s+temp=)[-\d\.]+"
    contenido = re.sub(patron_temp, rf"\g<1>{temp}", contenido)
    
    patron_vdd = r"(\.param\s+VDD\s*=\s*)[-\d\.]+"
    contenido = re.sub(patron_vdd, rf"\g<1>{vdd}", contenido)
    
    # ====================================================================
    # 2. REDIRIGIR LA SALIDA .RAW A LA CARPETA AISLADA DE LA ESQUINA
    #    (Se omite LDO_ac_estability y se incluye LDO_op para evitar colisiones)
    # ====================================================================
    nombres_base = [
        "LDO_tran_picos",
        "LDO_ac_psrr",
        "LDO_dc_line",
        "LDO_dc_load",
        "LDO_dc_dropout",
        "LDO_op"
    ]
    
    for base in nombres_base:
        ruta_absoluta_raw = os.path.join(sim_dir, f"{base}.raw")
        contenido = contenido.replace(f"{base}.raw", ruta_absoluta_raw)
    
    # Guardar netlist temporal en el directorio original para mantener rutas relativas
    netlist_temp_path = os.path.join(directorio_original, f"ldo_pex_temp_{nombre_llave}.spice")
    with open(netlist_temp_path, 'w') as archivo:
        archivo.write(contenido)

    # 3. EJECUTAR SIMULACIÓN EN NGSPICE (MODO BATCH)
    comando = ['/foss/tools/ngspice/bin/ngspice', "-b", netlist_temp_path]
    resultado = subprocess.run(comando, capture_output=True, text=True, cwd=directorio_original)
    
    if resultado.returncode != 0:
        print(f"\n⚠️ Falló la simulación en la esquina {nombre_llave}: {resultado.stderr[:200]}")
        if os.path.exists(netlist_temp_path):
            os.remove(netlist_temp_path)
        return

    # Inicializar diccionarios para esta esquina una vez verificada la simulación
    parametros_ldo[nombre_llave] = {}
    graficas_ldo[nombre_llave] = {}

    # Extraer LOG de variables medidas en consola por Ngspice
    mi_log = resultado.stdout
    patron = r"([a-zA-Z0-9_]+)\s*=\s*([+-]?\d+\.?\d*[eE]?[+-]?\d*)"
    coincidencias = re.findall(patron, mi_log)
    
    for nombre, valor in coincidencias:
        parametros_ldo[nombre_llave][nombre.strip().lower()] = float(valor)
    
    if "tnom" in parametros_ldo[nombre_llave]:
        del parametros_ldo[nombre_llave]["tnom"]

    # 4. EXTRAER LAS GRÁFICAS DESDE LOS ARCHIVOS RAW (PyLTSpice)
    try:
        raw_tran = RawRead(os.path.join(sim_dir, "LDO_tran_picos.raw"))
        raw_ac_psrr = RawRead(os.path.join(sim_dir, "LDO_ac_psrr.raw"))
        raw_dc_line = RawRead(os.path.join(sim_dir, "LDO_dc_line.raw"))
        raw_dc_load = RawRead(os.path.join(sim_dir, "LDO_dc_load.raw"))
        
        # Picos transitorios (Overshoot / Undershoot)
        graficas_ldo[nombre_llave]["time"] = raw_tran.get_trace("time").get_wave()
        graficas_ldo[nombre_llave]["t_vo"] = raw_tran.get_trace("v(vo)").get_wave()
        
        # PSRR (Frecuencia y magnitud extraídas directamente desde LDO_ac_psrr.raw)
        graficas_ldo[nombre_llave]["frecuencia"] = raw_ac_psrr.get_trace("frequency").get_wave()
        graficas_ldo[nombre_llave]["psrr"] = raw_ac_psrr.get_trace("psrr_db").get_wave()
        
        # Regulación de Línea (transitorio en rampa de Vin)
        graficas_ldo[nombre_llave]["line_time"] = raw_dc_line.get_trace("time").get_wave()
        graficas_ldo[nombre_llave]["line_vo"] = raw_dc_line.get_trace("v(vo)").get_wave()
        
        # Regulación de Carga (transitorio en rampa de Iload)
        graficas_ldo[nombre_llave]["load_time"] = raw_dc_load.get_trace("time").get_wave()
        graficas_ldo[nombre_llave]["load_vo"] = raw_dc_load.get_trace("v(vo)").get_wave()
        
    except Exception as e:
        print(f"\n⚠️ Error al leer los datos RAW de {nombre_llave}: {e}")

    # ====================================================================
    # 5. LIMPIEZA DEL NETLIST TEMPORAL
    # ====================================================================
    if os.path.exists(netlist_temp_path):
        os.remove(netlist_temp_path)
        
    # shutil.rmtree(sim_dir, ignore_errors=True) # Descomentar para borrar los raws


In [3]:
# ==============================================================================
# Celda 3: Lanzador Multinúcleo (Paralelismo) y Reporte de Resultados PEX
# Descripción:
#   Define las combinaciones de temperatura, voltaje y esquinas de proceso,
#   ejecuta las simulaciones PEX en paralelo y genera las tablas resumen.
# ==============================================================================
temperaturas = [-40, 27, 125]
voltajes = [1.62, 1.80, 1.98]

corners_proceso = [
    "tt",  # Caso nominal
    "ss",
    "ff",
    "sf",
    "fs",  # 4 esquinas base de transistores
    "fs_ll",  # Peor caso real de PSRR, Load Reg y Line Reg
    "sf_ll",  # Peor caso real de Overshoot
    "ss_hh",  # Peor caso de Dropout y mínima corriente en self_bias (R alta)
    "ff_lh",  # Peor caso real de Undershoot
]
"""
temperaturas = [-40, 27, 125]          # Temperaturas en grados Celsius [°C]
voltajes = [1.62, 1.80, 1.98]          # Niveles de alimentación VDD [V]

# Esquinas de proceso estándar principales
corners_proceso = [
    "tt", "sf", "ff", "ss", "fs", "ll", "hh", "hl", "lh",
    "sf_ll", "sf_hh", "sf_hl", "sf_lh", "ff_ll", "ff_hh", "ff_hl", "ff_lh",
    "ss_ll", "ss_hh", "ss_hl", "ss_lh", "fs_ll", "fs_hh", "fs_hl", "fs_lh"
]
"""
# Ruta al netlist generado por Xschem desde 'ldo_tb_pex.sch'
netlist_path = "/headless/.xschem/simulations/ldo_tb_pex.spice"
# Carpeta separada para los archivos .raw de PEX (no altera los .raw del esquemático)
raw_path = "/foss/designs/LDO-sky130/xschem/spice_raw_log/raw_pex/"
os.makedirs(raw_path, exist_ok=True)

todas_las_combinaciones = list(itertools.product(temperaturas, voltajes, corners_proceso))
total_sim = len(todas_las_combinaciones)

print(f"Lanzando {total_sim} simulaciones PEX en PARALELO...")

with concurrent.futures.ThreadPoolExecutor(max_workers=4) as executor:
    futuros = [
        executor.submit(one_pvt_parallel, netlist_path, raw_path, temp, vdd, proces)
        for temp, vdd, proces in todas_las_combinaciones
    ]
    
    completados = 0
    for futuro in concurrent.futures.as_completed(futuros):
        completados += 1
        print(f"Progreso global: {completados}/{total_sim} completados.", end='\r')

print("\n\n¡Simulación masiva paralela PEX finalizada!")

# ==============================================================================
# 4. REPORTES NUMÉRICOS Y EXPORTACIÓN A EXCEL / CSV
# ==============================================================================
df_resultados = pd.DataFrame.from_dict(parametros_ldo, orient='index')

if not df_resultados.empty and len(df_resultados.columns) > 0:
    df_resumen = df_resultados.agg(['min', 'max', 'mean']).T

    print("--- Resumen PVT (PEX): Peor y Mejor Caso de cada Especificación ---")
    try:
        display(df_resumen)
    except NameError:
        print(df_resumen)

    try:
        with pd.ExcelWriter("Reporte_Completo_LDO_PEX.xlsx") as writer:
            df_resultados.to_excel(writer, sheet_name="225_Corners_PEX")
            df_resumen.to_excel(writer, sheet_name="Resumen_Peores_Casos_PEX")
        print("\n¡Reporte exportado exitosamente a 'Reporte_Completo_LDO_PEX.xlsx'!")
    except ImportError:
        print("\nLibrería de Excel no detectada. Guardando como CSV...")
        df_resultados.to_csv("225_Corners_LDO_PEX.csv")
        df_resumen.to_csv("Resumen_Peores_Casos_LDO_PEX.csv")
        print("¡Tablas exportadas exitosamente a formato CSV!")
else:
    print("⚠️ No se obtuvieron resultados válidos. Verifica que '/headless/.xschem/simulations/ldo_tb_pex.spice' haya sido generado con el botón Netlist en Xschem.")


Lanzando 81 simulaciones PEX en PARALELO...
Progreso global: 81/81 completados.

¡Simulación masiva paralela PEX finalizada!
--- Resumen PVT (PEX): Peor y Mejor Caso de cada Especificación ---


,min,max,mean
temp,-40.000000,125.000000,37.333333
gananciapsrr_1,18.264800,111.114700,64.591914
gananciapsrr_1k,18.293800,82.137800,62.359446
gananciapsrr_10k,20.634670,62.539210,53.669504
gananciapsrr_1m,57.116330,75.582920,64.426623
voline_max,-1.006990,1.281810,1.175458
at,0.000230,0.000437,0.000336
voline_min,-1.006990,1.199980,1.169295
salida_line,-1.006990,1.213688,1.171133
from,0.000150,0.000150,0.000150



Librería de Excel no detectada. Guardando como CSV...
¡Tablas exportadas exitosamente a formato CSV!


In [4]:
# ==============================================================================
# Celda 4: Exportación manual a CSV con nombres PEX
# ==============================================================================
print("\nGuardando tablas de resultados PEX como CSV...")
df_resultados.to_csv("225_Corners_LDO_PEX.csv")
df_resumen.to_csv("Resumen_Peores_Casos_LDO_PEX.csv")
print("¡Tablas PEX exportadas exitosamente a formato CSV!")



Guardando tablas de resultados PEX como CSV...
¡Tablas PEX exportadas exitosamente a formato CSV!


In [5]:
# ==============================================================================
# Celda 5: Graficador de todas las curvas PEX (PSRR, Transitorio, Línea y Carga)
# Descripción:
#   Genera y guarda las gráficas superpuestas de todas las esquinas PVT para PEX
#   con sufijo '_pex.png' (sin gráfica de estabilidad en lazo abierto).
# ==============================================================================
print("\nGenerando gráficas PEX...")

# --- 1. PSRR ---
plt.figure(figsize=(10, 6))
exitosas_psrr = 0
for llave, curvas in graficas_ldo.items():
    if "frecuencia" in curvas and "psrr" in curvas:
        x = np.abs(curvas["frecuencia"])
        y = np.real(curvas["psrr"])
        min_len = min(len(x), len(y))
        plt.semilogx(x[:min_len], y[:min_len], linewidth=0.8, alpha=0.3)
        exitosas_psrr += 1
        
plt.title(f"Análisis PVT (PEX): PSRR del LDO (Curvas: {exitosas_psrr})")
plt.xlabel("Frecuencia [Hz]")
plt.ylabel("PSRR [dB]")
plt.grid(True, which="both", linestyle="--", linewidth=0.5)
plt.savefig("psrr_pex.png", dpi=300, bbox_inches='tight')
plt.close()

# --- 2. RESPUESTA TRANSITORIA (Overshoot y Undershoot) ---
plt.figure(figsize=(10, 6))
exitosas_tran = 0
for llave, curvas in graficas_ldo.items():
    if "time" in curvas and "t_vo" in curvas:
        x = np.real(curvas["time"])
        y = np.real(curvas["t_vo"])
        min_len = min(len(x), len(y))
        plt.plot(x[:min_len], y[:min_len], linewidth=0.8, alpha=0.3)
        exitosas_tran += 1
        
plt.title(f"Análisis PVT (PEX): Respuesta Transitoria (Curvas: {exitosas_tran})")
plt.xlabel("Tiempo [s]")
plt.ylabel("Voltaje de Salida (Vo) [V]")
plt.grid(True, linestyle="--", linewidth=0.5)
plt.savefig("tran_pex.png", dpi=300, bbox_inches='tight')
plt.close()

# --- 3. REGULACIÓN DE LÍNEA (Rampa) ---
plt.figure(figsize=(10, 6))
exitosas_line = 0
for llave, curvas in graficas_ldo.items():
    if "line_time" in curvas and "line_vo" in curvas:
        x = np.real(curvas["line_time"])
        y = np.real(curvas["line_vo"])
        min_len = min(len(x), len(y))
        plt.plot(x[:min_len], y[:min_len], linewidth=0.8, alpha=0.3)
        exitosas_line += 1
        
plt.title(f"Análisis PVT (PEX): Regulación de Línea (Curvas: {exitosas_line})")
plt.xlabel("Tiempo de Rampa [s] (Simula Vin)")
plt.ylabel("Voltaje de Salida (Vo) [V]")
plt.grid(True, linestyle="--", linewidth=0.5)
plt.savefig("grafica_regulacion_linea_pex.png", dpi=300, bbox_inches='tight')
plt.close()

# --- 4. REGULACIÓN DE CARGA (Rampa) ---
plt.figure(figsize=(10, 6))
exitosas_load = 0
for llave, curvas in graficas_ldo.items():
    if "load_time" in curvas and "load_vo" in curvas:
        x = np.real(curvas["load_time"])
        y = np.real(curvas["load_vo"])
        min_len = min(len(x), len(y))
        plt.plot(x[:min_len], y[:min_len], linewidth=0.8, alpha=0.3)
        exitosas_load += 1
        
plt.title(f"Análisis PVT (PEX): Regulación de Carga (Curvas: {exitosas_load})")
plt.xlabel("Tiempo de Rampa [s] (Simula Iload)")
plt.ylabel("Voltaje de Salida (Vo) [V]")
plt.grid(True, linestyle="--", linewidth=0.5)
plt.savefig("grafica_regulacion_carga_pex.png", dpi=300, bbox_inches='tight')
plt.close()

print("Todas las gráficas PEX (*_pex.png) se guardaron en la carpeta actual.")



Generando gráficas PEX...
Todas las gráficas PEX (*_pex.png) se guardaron en la carpeta actual.


In [6]:
# ==============================================================================
# Celda 6: Respaldo de diccionarios de resultados y curvas PEX en formato Pickle
# ==============================================================================
import pickle
import os

def guardar_resultados_pex():
    carpeta_destino = "resultados_simulacion"
    os.makedirs(carpeta_destino, exist_ok=True)
    
    # Guardar los parámetros escalares medidos en PEX
    ruta_parametros = os.path.join(carpeta_destino, "parametros_ldo_pex0ma.pkl")
    with open(ruta_parametros, 'wb') as archivo:
        pickle.dump(parametros_ldo, archivo)
        
    # Guardar los vectores de las gráficas medidas en PEX
    ruta_graficas = os.path.join(carpeta_destino, "graficas_ldo_pex0ma.pkl")
    with open(ruta_graficas, 'wb') as archivo:
        pickle.dump(graficas_ldo, archivo)
        
    print(f"✅ Datos PEX guardados exitosamente en la carpeta: {carpeta_destino}")

# Llamar a la función al finalizar las simulaciones
guardar_resultados_pex()


✅ Datos PEX guardados exitosamente en la carpeta: resultados_simulacion
